# `swarmtorch` — full standardised benchmark suite (Kaggle GPU)

Runs every paper-grade benchmark end-to-end on Kaggle. **Don't run this locally**; each
cell streams output so you can watch progress and read each `report.md` as it lands.

**Kaggle settings before you run anything:**
1. Right-pane → **Settings** → **Accelerator: GPU T4 x2** (or P100/V100).
2. **Internet: On** (needed for the `git clone` and MNIST/CIFAR downloads).
3. **Save Version → Save & Run All (Commit)** if you want the run to survive your
   tab closing.

**Estimated total runtime on T4 ≈ 4–6 hours** (well within Kaggle's 12-hour limit).

Every cell that runs a benchmark:
- streams the orchestrator output so you see live progress;
- previews the resulting `report.md` inline (machine fingerprint, mean ± std table,
  Friedman + Nemenyi block);
- saves per-seed JSONs under `results/<sweep>/` (each JSON carries the hardware
  fingerprint + git SHA in `meta.hardware`).

Final cell zips everything to `/kaggle/working/swarmtorch_results_<timestamp>.zip`
for download.

---

### Benchmark coverage

| Stage | Sweep | What it tests |
|---|---|---|
| 4.1 | `bench-synthetic` | Dimensionality-wall study: 5 CEC functions × d∈{10,50,200,1000} × 13 algos × 10 seeds |
| 4.4 | `ablation-init` | Kaiming-aware swarm init vs uniform/gaussian (falsifies Stage 1.1 fix) |
| 4.4 | `ablation-swarm` | Population-size sweep {10, 30, 100, 300} |
| 4.3 | `bench-hpo` | Metaheuristic searchers vs Random / TPE / Hyperband |
| 6   | `bench-gpu` | **The headline:** swarmtorch GPU vmap vs NumPy reference |
| 4.2 | `bench-training` | Real NNs — MNIST 2/4-layer MLP, CIFAR-10 small CNN, tabular regression |

**Known gaps (flagged for the paper, not blocking):** no external-library baseline
(pyMetaheuristic / Nevergrad), no equal-wall-clock view for gradient methods, CIFAR
CNN is ~1M params (we can't fit a transformer in T4's 12-hour budget), no per-
operator-group aggregation in the auto-report.

## 1. Clone + install + identify the machine

This is the one cell that has to succeed before anything else runs. If you see
`CUDA available: True` and `gpu : Tesla T4` (or similar) below, the rest will work.

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/hallelx2/swarmtorch.git'
if not os.path.isdir('swarmtorch'):
    subprocess.check_call(['git', 'clone', REPO_URL])
%cd swarmtorch
subprocess.check_call(['git', 'log', '--oneline', '-1'])

# Install with benchmark + cmaes extras. PyTorch and CUDA already ship on Kaggle.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[benchmark,cmaes]'])

from swarmtorch.benchmark import print_banner
print_banner()

## 2. Smoke test (~1 minute)

Tiny run covering both the synthetic harness and the GPU path so we fail fast on a
broken environment instead of 4 hours into the real run.

In [ ]:
!python scripts/cloud_bench.py --quick --include-gpu

## 3. Helper — preview a `report.md` inline

Used by every benchmark cell below so you can read the result table as soon as
the sweep finishes, without leaving the notebook.

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

def show_report(path):
    p = Path(path)
    if not p.exists():
        print(f'!! no report at {p}')
        return
    display(Markdown(p.read_text(encoding='utf-8')))

def show_pngs(dir_):
    from IPython.display import Image
    for png in sorted(Path(dir_).glob('*.png')):
        print(png.name)
        display(Image(filename=str(png)))

## 4. Stage 4.4 — Init-strategy ablation (~15 min)

Falsifiable test of the Stage 1.1 Kaiming-aware init claim. We sweep PSO / CA / TLBO /
CMA-ES across `init_strategy ∈ {model, uniform, gaussian}` on Rastrigin and Rosenbrock at
d=50 with 5 seeds. If the `model` init does NOT beat `uniform`, the Kaiming fix is
either wrong or unnecessary.

**Run this first because it's the shortest and validates the most important code claim.**

In [ ]:
!python scripts/run_ablations.py \
    --ablation init_strategy \
    --output-dir results/ablations/init \
    --seeds 0 1 2 3 4 \
    --algorithms PSO CA TLBO CMAES

show_report('results/ablations/init/report.md')
show_pngs('results/ablations/init')

## 5. Stage 4.4 — Swarm-size ablation (~20 min)

PSO and CMA-ES at swarm sizes {10, 30, 100, 300} on Rastrigin + Rosenbrock at d=50.
Identifies the population-size sweet spot and indirectly demonstrates the GPU value
(at swarm=300 the vmap fast path matters most).

In [ ]:
!python scripts/run_ablations.py \
    --ablation swarm_size \
    --output-dir results/ablations/swarm \
    --seeds 0 1 2 3 4 \
    --algorithms PSO CMAES

show_report('results/ablations/swarm/report.md')
show_pngs('results/ablations/swarm')

## 6. Stage 4.3 — HPO comparison (~30 min)

Metaheuristic searchers vs Random / TPE / Hyperband baselines on three HPO tasks:
small CNN (4 continuous params), tiny transformer (continuous + categorical), XGBoost
tabular (mixed). Exercises the categorical encoding fix from Stage 1.6.

In [ ]:
!python scripts/run_hpo.py \
    --output-dir results/hpo \
    --seeds 0 1 2 3 4 \
    --n-trials 20

show_report('results/hpo/report.md')

## 7. Stage 6 — GPU vs NumPy headline (~30 min) ⭐

**The headline figure of the paper.** swarmtorch CUDA vmap vs swarmtorch CPU vmap vs
swarmtorch CPU loop vs pure-NumPy reference, on Sphere / Rastrigin / Ackley at d ∈
{100, 1000} with swarm sizes {64, 256, 1024}. Each cell is timed three times with a
warm-up.

Expected speedup on T4: 10–100× at swarm_size ≥ 256.

In [ ]:
!python scripts/run_gpu_vs_numpy.py \
    --output-dir results/gpu_vs_numpy \
    --functions sphere rastrigin ackley \
    --dims 100 1000 \
    --swarm-sizes 64 256 1024 \
    --seeds 0 1 \
    --max-fe 20000

show_report('results/gpu_vs_numpy/report.md')

### 7b. Speedup plot — the figure that goes in the paper

In [ ]:
import json, glob
from collections import defaultdict
import matplotlib.pyplot as plt

records = [json.load(open(p)) for p in sorted(glob.glob('results/gpu_vs_numpy/*.json'))]
g = defaultdict(list)
for r in records:
    g[(r['function'], r['dim'], r['variant'])].append((r['swarm_size'], r['wall_seconds']))

funcs = sorted({(f, d) for (f, d, _) in g})
fig, axes = plt.subplots(1, len(funcs), figsize=(5 * len(funcs), 4), squeeze=False, dpi=120)
for ax, (func, dim) in zip(axes[0], funcs):
    for variant in ['numpy', 'swarmtorch-cpu-loop', 'swarmtorch-cpu-vmap', 'swarmtorch-cuda-vmap']:
        if (func, dim, variant) not in g: continue
        from collections import defaultdict as dd
        avg = dd(list)
        for ss, w in g[(func, dim, variant)]:
            avg[ss].append(w)
        xs = sorted(avg)
        ys = [sum(avg[x]) / len(avg[x]) for x in xs]
        ax.plot(xs, ys, marker='o', label=variant)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('swarm size'); ax.set_ylabel('wall-clock seconds')
    ax.set_title(f'{func}, d={dim}'); ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig('results/gpu_vs_numpy/speedup_curves.png', bbox_inches='tight')
plt.show()
print('Saved: results/gpu_vs_numpy/speedup_curves.png')

## 8. Stage 4.1 — Synthetic dimensionality-wall study (~1.5 hours)

5 CEC functions × **d ∈ {10, 50, 200, 1000}** × 13 algorithms × 10 seeds = 2,600 cells.

This is the paper's most defensible empirical contribution. d=1000 is the CEC standard
ceiling — without it, reviewers will ask "how do you know your dim-wall claim
generalizes?"

Friedman + Nemenyi appear automatically in the report.

In [ ]:
!python scripts/run_synthetic.py \
    --output-dir results/synthetic \
    --seeds 0 1 2 3 4 5 6 7 8 9 \
    --max-fe 5000 \
    --dims 10 50 200 1000

show_report('results/synthetic/report.md')
show_pngs('results/synthetic')

## 9. Stage 4.2 — Real NN training (~2–3 hours)

MNIST 2-layer MLP + MNIST 4-layer MLP + CIFAR-10 small CNN + tabular regression, with
all 13 paper algorithms at FE=3000 across 5 seeds. Downloads MNIST + CIFAR on first run
via torchvision.

**Limitation:** the CIFAR CNN is ~1M params; transformer/large-CNN claims need a longer
session than Kaggle's 12-hour budget allows.

In [ ]:
import subprocess, sys
# torchvision is preinstalled on Kaggle; if not, install it.
try:
    import torchvision  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchvision'])

!python scripts/run_training.py \
    --output-dir results/training \
    --seeds 0 1 2 3 4 \
    --max-fe 3000

show_report('results/training/report.md')
show_pngs('results/training')

## 10. Bundle everything for download

Right pane → **Output** tab → download `swarmtorch_results_<timestamp>.zip`. Every
JSON in the bundle has the machine fingerprint (CPU model, GPU model, RAM, OS, Python
+ PyTorch versions, git commit SHA) under `meta.hardware`.

In [ ]:
import shutil, time
stamp = time.strftime('%Y%m%d_%H%M')
out = f'/kaggle/working/swarmtorch_results_{stamp}'
shutil.make_archive(out, 'zip', root_dir='.', base_dir='results')
print(f'Bundle ready: {out}.zip')

# Convenience link in the notebook output:
try:
    from IPython.display import FileLink
    display(FileLink(f'{out}.zip'))
except Exception:
    pass

---

## Done

Six benchmarks ran end-to-end. The bundled zip contains:

* `results/synthetic/` — 2,600 cells, dimensionality-wall report + 5 convergence PNGs.
* `results/training/` — MNIST + CIFAR + tabular, 4 tasks × 13 algos × 5 seeds.
* `results/hpo/` — HPO comparison report.
* `results/ablations/init/` + `results/ablations/swarm/` — falsifiable ablations.
* `results/gpu_vs_numpy/` — headline speedup table + speedup_curves.png.

Every result JSON carries the machine fingerprint under `meta.hardware` (CPU model,
GPU model + memory, RAM, OS, Python + PyTorch + CUDA versions, git commit SHA), so
the reports stay traceable to exactly which Kaggle session produced them.